# ME324 · Lab 6 — Classifying images (CNNs)

**Lecture 6 · "Classifying images" · 2026-08-10**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsrobinson/me324/blob/main/labs/lab-06-cnn-images.ipynb)

**How this notebook works.** Cells marked `# TODO` are for you to fill in — each one is
short and clearly marked. Worked answers for all of them are in the **Solutions** section
at the very bottom, so you will never be stuck for long. Have a real go first, then check.

**And please use an LLM.** ChatGPT, Claude, Gemini, DeepSeek — whichever you like.
*"In PyTorch, how do I …?"* is exactly the kind of question these tools are excellent at, and
looking things up this way is what every working researcher does. Two habits worth keeping:
ask for the **explanation** rather than just the line, and **run everything** it hands you.
The exam is closed-book, so what counts is that you can read the code back and say what it
does.

---

> **Turn the GPU on first** — Colab: **Runtime → Change runtime type → T4 GPU**. To process images, we'll need more compute -- the T4 GPU's are free to use on colab.

### Today's goal

Build, train, and look inside a small **convolutional neural network (CNN)** that
classifies clothing images (**FashionMNIST**) into 10 categories. By the end you can:

1. Explain the **parameter explosion** that rules out MLPs for images
2. Build and train a CNN, predicting its **shapes** and **parameter counts** by hand
3. Read a trained model's filters, feature maps and **confusion matrix**.

## ⏱️ Plan for today (~90 minutes)

This lab is built for a single 90-minute session, and it's **completely fine not to finish every cell in the room**.

- **Core — do these:** Sections 1–3 — the output-size formula, the CNN, the training loop.
- **Stretch / take-home — skip if short on time:** Section 4 — predictions, the confusion matrix, filters, the MLP baseline.

_There are **seven** `# TODO` cells today. Worked answers are in the **Solutions** section at the bottom._

## Run me first

Imports, the course seed (1337), and the **device**.

In [ ]:
# If a package is missing (e.g. running locally), uncomment the next line:
# !pip install torch torchvision matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

# Reproducibility: same seed -> same shuffles, same initial weights.
torch.manual_seed(1337)

# CNNs love a GPU. 'cuda' = an NVIDIA GPU; 'cpu' = your processor.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("torch version:", torch.__version__)
print("Using device :", device)


## The data — FashionMNIST

FashionMNIST is a drop-in replacement for the classic MNIST digits: **70,000 grayscale
images**, 28×28 pixels, one of **10 clothing categories** each, split 60,000 train /
10,000 test. An image is just a tensor of numbers: `ToTensor()` rescales the raw 0–255
pixels to [0, 1], giving each image shape **(1, 28, 28)** = `(channels, height, width)`.
A **`DataLoader`** will later serve shuffled **batches**.

In [ ]:
# Download FashionMNIST (cached after the first run) and convert images to tensors.
train_data = datasets.FashionMNIST(root="data", train=True,  download=True, transform=ToTensor())
test_data  = datasets.FashionMNIST(root="data", train=False, download=True, transform=ToTensor())

# The 10 integer labels (0..9) map to these human-readable names:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("training images:", len(train_data))
print("test images    :", len(test_data))

img, label = train_data[0]
print("one image tensor shape:", tuple(img.shape))         # (1, 28, 28) = (channels, H, W)
print("pixel value range:", float(img.min()), "to", float(img.max()))
print("its label:", label, "=", class_names[label])


In [ ]:
# A DataLoader serves the data in BATCHES. shuffle=True reshuffles every epoch.
BATCH_SIZE = 64
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

# Peek at a single batch:
images, labels = next(iter(train_loader))
print("one batch of images:", tuple(images.shape))   # (B, C, H, W) = (64, 1, 28, 28)
print("one batch of labels:", tuple(labels.shape))   # (64,)


### Always look at your data

Before modelling anything, eyeball a few examples.

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(9, 5))
for ax, img, label in zip(axes.flat, images, labels):
    ax.imshow(img.squeeze(), cmap="gray")        # .squeeze() drops the channel dim for plotting
    ax.set_title(class_names[label], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 1 · Why not just use an MLP?

In Lab 5 you built a fully-connected network (an **MLP**). Why not flatten each image
into a long vector and do the same here?

Because of the **parameter explosion**: the lecture's 256×256 RGB example needs ~25
**million** weights in its first hidden layer alone!
Flattening the images would also ignore **spatial structure** and **translation invariance** (e.g. a shoe
is a shoe wherever it sits in the frame).

We can solve these issues in one go by using **convolutions**: a small kernel (say 3×3) that slides across the whole image,
reusing the *same* weights at every location.

In [ ]:
# --- The lecture's worked example: a 256x256 RGB image into a plain MLP ---
H, W, C = 256, 256, 3
n_inputs = H * W * C
hidden = 128
mlp_first_layer = n_inputs * hidden
print(f"256x256 RGB flattened : {n_inputs:,} input features")
print(f"...into 128 neurons   : {mlp_first_layer:,} weights  (~{mlp_first_layer/1e6:.1f} million)")

# --- Same idea for OUR data: 28x28 grayscale FashionMNIST ---
fm_inputs = 28 * 28
print(f"\nFashionMNIST flattened: {fm_inputs:,} input features")
print(f"...into 128 neurons   : {fm_inputs * hidden:,} weights")

# --- A convolution instead: one 3x3 layer with 32 filters (the lecture's 'stop & check') ---
conv_params = (3 * 3 * 1 + 1) * 32           # (kernel + 1 bias) per filter
fc_equiv    = 28 * 28 * 32                    # what a fully-connected layer making 32 maps costs
print(f"\n3x3 conv, 32 filters  : {conv_params:,} parameters")
print(f"fully-connected equiv : {fc_equiv:,} weights  ({fc_equiv // conv_params}x more)")
print("\nParameter sharing: the SAME 3x3 kernel is reused at every pixel position.")


## 2 · Build the CNN

Today you will build a CNN with two **convolution blocks** to extract features, then a
**fully-connected head** to turn them into 10 class scores:

```
Conv → ReLU → MaxPool   (×2)   →   Flatten → Linear → ReLU → Linear
```

The **output-size formula** gives the spatial size after any conv or pool ($H$ = input
height, $k$ = kernel size, $P$ = padding, $s$ = stride; width follows the same rule):

$$\text{out} = \left\lfloor \frac{H + 2P - k}{s} \right\rfloor + 1$$

One image's journey through our network:

| Layer | Output shape (C×H×W) | Why |
|---|---|---|
| input | 1 × 28 × 28 | grayscale image |
| Conv1 (1→16, 3×3, pad 1) | 16 × 28 × 28 | ⌊(28+2·1−3)/1⌋+1 = 28 |
| MaxPool (2×2, stride 2) | 16 × 14 × 14 | ⌊(28−2)/2⌋+1 = 14 |
| Conv2 (16→32, 3×3, pad 1) | 32 × 14 × 14 | ⌊(14+2·1−3)/1⌋+1 = 14 |
| MaxPool (2×2, stride 2) | 32 × 7 × 7 | ⌊(14−2)/2⌋+1 = 7 |
| Flatten | 1568 | 32 × 7 × 7 = 1568 |
| Linear (1568→128) | 128 | fully-connected |
| Linear (128→10) | 10 | one score per class |

**Padding = 1** keeps the 3×3 convs size-preserving ("same" padding); the pooling does
the shrinking. First, drive the formula yourself on a layer the table does *not*
cover.

In [ ]:
# out = (H + 2P - k) // s + 1, applied to height and width alike.
# Suppose we had picked a 5x5 kernel with NO padding for the first conv.
# TODO 1: what spatial size comes out of that conv on a 28x28 image?  (k=5, P=0, s=1)
# TODO 2: ...and after the 2x2 max-pool, stride 2?                    (k=2, P=0, s=2)
after_conv = None  # <-- TODO
after_pool = None  # <-- TODO

# Now let PyTorch mark your answers:
x = torch.zeros(1, 1, 28, 28)
conv_out = nn.Conv2d(1, 8, kernel_size=5, padding=0)(x)
pool_out = nn.MaxPool2d(kernel_size=2, stride=2)(conv_out)
print("you predicted:", after_conv, "then", after_pool)
print("PyTorch says :", conv_out.shape[-1], "then", pool_out.shape[-1])

Now build the network as an `nn.Module`: implement **both** `__init__` and `forward`
against the docstring's spec and the shape table. Everything that follows runs on this
class.

In [ ]:
class CNN(nn.Module):
    """A small CNN for FashionMNIST.

    Define these layers in __init__ (use exactly these attribute names):
        conv1 : Conv2d   1 -> 16 channels, 3x3 kernel, padding 1
        conv2 : Conv2d  16 -> 32 channels, 3x3 kernel, padding 1
        pool  : MaxPool2d  2x2, stride 2
        fc1   : Linear  (32 * 7 * 7) -> 128
        fc2   : Linear  128 -> 10

    forward: for each conv block do  conv -> ReLU -> pool, then flatten and run the head.
        input            (B, 1, 28, 28)
          conv1+relu+pool -> (B, 16, 14, 14)
          conv2+relu+pool -> (B, 32,  7,  7)
          flatten         -> (B, 1568)
          fc1 + relu      -> (B, 128)
          fc2 (logits)    -> (B, 10)     # raw logits, NO softmax
    """
    def __init__(self):
        super().__init__()
        # TODO: define conv1, conv2, pool, fc1, fc2 exactly as in the spec above.
        raise NotImplementedError("Define the CNN layers in __init__")

    def forward(self, x):
        # TODO: implement the forward pass described in the spec above.
        raise NotImplementedError("Implement the forward pass")


### Watch the shapes change

Push one real batch through, layer by layer, and check each shape against the table —
predicting shapes by hand is the single best way to avoid CNN bugs.

In [ ]:
model = CNN().to(device)
print(model)
print()

# Push one batch through, one layer at a time, and watch the shape change.
x = images.to(device)                                  # (64, 1, 28, 28)
print("input          ", tuple(x.shape))
x = model.conv1(x);          print("after conv1    ", tuple(x.shape), " 1->16 ch, 3x3 pad1: (28+2-3)/1+1 = 28")
x = model.pool(F.relu(x));   print("after relu+pool", tuple(x.shape), " 2x2 pool: (28-2)/2+1 = 14")
x = model.conv2(x);          print("after conv2    ", tuple(x.shape), " 16->32 ch, 3x3 pad1: (14+2-3)/1+1 = 14")
x = model.pool(F.relu(x));   print("after relu+pool", tuple(x.shape), " 2x2 pool: (14-2)/2+1 = 7")
x = torch.flatten(x, 1);     print("after flatten  ", tuple(x.shape), " 32*7*7 = 1568")


### Count the parameters (with biases)

A conv layer holds `(kernel_h × kernel_w × in_channels + 1) × out_channels` parameters, independent of image size. 
A `Linear(in, out)` holds `in × out + out`. 
Work each layer out by hand, and notice *where* the parameters live.

In [ ]:
def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

# TODO: plain arithmetic, straight from the formulas above — e.g. conv1 is a 3x3
# kernel over 1 input channel, with 16 filters.
conv1_expected = None  # <-- TODO
conv2_expected = None  # <-- TODO
fc1_expected   = None  # <-- TODO  (1568 in-features -> 128)
fc2_expected   = None  # <-- TODO  (128 -> 10)

for name, layer, expected in [("conv1", model.conv1, conv1_expected),
                              ("conv2", model.conv2, conv2_expected),
                              ("fc1",   model.fc1,   fc1_expected),
                              ("fc2",   model.fc2,   fc2_expected)]:
    actual = count_parameters(layer)
    verdict = "correct" if expected == actual else "check your arithmetic"
    print(f"{name:6s} you said {str(expected):>8s} | PyTorch counts {actual:>7,d}   <- {verdict}")
print("-" * 40)
print(f"TOTAL  {count_parameters(model):,}")

## 3 · Train it — the Lab 5 loop, now on batches

The training loop is the same as **Lab 5**:

```
zero_grad → forward → loss → backward → step
```

New here: we loop over **batches** from the `DataLoader` (one full pass over the
training set is an **epoch**), and each batch moves to the **`device`** with
`.to(device)`. If your model and data are not on the same device, Pytorch will error.

The loss is **cross-entropy** (`nn.CrossEntropyLoss`): raw logits and integer labels in,
softmax applied internally. Like in Lab 5, as we work directly with the logits, we should not
apply softmax in the model.

First, finish defining the below helper to measure **accuracy** — the fraction of images classified correctly.

In [ ]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval()                          # evaluation mode
    correct, total = 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)                 # (B, 10) — one score per class
        preds = ...                       # TODO 1: the predicted class per image (highest score)
        correct += ...                    # TODO 2: how many predictions match y — a plain Python int
        total   += y.size(0)
    model.train()                         # back to training mode
    return correct / total

Now the loop itself — the same five steps, wrapped in loops over epochs and batches:

In [ ]:
def train(model, train_loader, epochs=3, lr=1e-3):
    """Train `model` and return the list of per-batch losses.

    For each epoch, for each batch (X, y) from train_loader:
        move the batch to `device`, run the five Lab 5 steps
        (zero_grad -> forward -> loss -> backward -> step),
        and append loss.item() to loss_history.
    After each epoch, print the epoch number, the last loss, and the train and
    test accuracy (use the accuracy() helper), so you can watch learning happen.
    """
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_history = []
    # TODO: write the epoch loop, the batch loop, and the five-step body.

    return loss_history

Now train. **Three epochs** gets past ~88% test accuracy in a minute or two on a GPU;
more epochs buy a few more points.

In [ ]:
loss_history = train(model, train_loader, epochs=3, lr=1e-3)


### The loss curve

The per-batch loss should fall fast, then flatten — the classic learning curve.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel("training step (batch)")
plt.ylabel("cross-entropy loss")
plt.title("Training loss")
plt.grid(alpha=0.3)
plt.show()

print(f"Final train accuracy: {accuracy(model, train_loader):.3f}")
print(f"Final test  accuracy: {accuracy(model, test_loader):.3f}")


## 4 · Look inside the network

Contrary to popular opinion, a trained model is not a black box. We absolutely can inspect the predictions, mistakes, filters, feature maps of our CNN!

### Predictions — right and wrong

Take one test batch and make the predictions:

In [ ]:
model.eval()
X, y = next(iter(test_loader))
with torch.no_grad():
    # TODO: run the batch through the model and take the highest-scoring class per
    # image, ending up with a tensor on the CPU (matplotlib cannot read GPU tensors).
    # Remember: the model lives on `device`; X does not (yet).
    preds = ...

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for ax, img, true, pred in zip(axes.flat, X, y, preds):
    ax.imshow(img.squeeze(), cmap="gray")
    is_right = (true == pred)
    ax.set_title(f"pred: {class_names[pred]}\ntrue: {class_names[true]}",
                 color=("green" if is_right else "red"), fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Where it goes wrong

You should find the model most often confuses visually similar classes —
shirts, T-shirts, coats, pullovers". 

In [ ]:
wrong_imgs, wrong_true, wrong_pred = [], [], []
model.eval()
with torch.no_grad():
    for X, y in test_loader:
        p = model(X.to(device)).argmax(dim=1).cpu()
        mask = p != y
        for img, t, pr in zip(X[mask], y[mask], p[mask]):
            wrong_imgs.append(img); wrong_true.append(t); wrong_pred.append(pr)
        if len(wrong_imgs) >= 12:
            break

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for ax, img, t, pr in zip(axes.flat, wrong_imgs, wrong_true, wrong_pred):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"pred: {class_names[pr]}\ntrue: {class_names[t]}", color="red", fontsize=7)
    ax.axis("off")
plt.suptitle("Where the CNN goes wrong", y=1.03)
plt.tight_layout()
plt.show()


### The confusion matrix

Let's calculate a confusion matrix for our classes. Entry ($i$, $j$)
counts test images of true class $i$ predicted as class $j$, so a perfect model will only have values on the
diagonal. Use this confusion matrix to find the worst confusion

In [ ]:
# Count every (true, predicted) pair on the test set into a 10x10 matrix.
confusion = torch.zeros(10, 10, dtype=torch.int64)
model.eval()
with torch.no_grad():
    for X, y in test_loader:
        p = model(X.to(device)).argmax(dim=1).cpu()
        for t, pr in zip(y, p):
            confusion[t, pr] += 1

plt.figure(figsize=(6.5, 5.5))
plt.imshow(confusion, cmap="Blues")
plt.xticks(range(10), class_names, rotation=90, fontsize=7)
plt.yticks(range(10), class_names, fontsize=7)
plt.xlabel("predicted"); plt.ylabel("true")
plt.colorbar(label="count")
plt.title("Test-set confusion matrix")
plt.tight_layout()
plt.show()

# TODO: find the single worst confusion — the largest OFF-diagonal entry — and
# print the two class names involved. Hints: .clone() the matrix and zero the
# diagonal with .fill_diagonal_(0); .argmax() on a 2-D tensor returns a FLAT
# index.

### The learned first-layer filters

Most of the sixteen 3×3 kernels `conv1` learned **from data** look messy: each is tuned to whatever helped
the loss. Notice this is quite different from a "theoretical" image kernel like a gaussian blur, and again
speaks to the lack of "interpretability" of model parameters in deep networks.

In [ ]:
filters = model.conv1.weight.data.cpu()     # shape (16, 1, 3, 3)
print("conv1 weight shape:", tuple(filters.shape), "= (out_channels, in_channels, kH, kW)")

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, f in zip(axes.flat, filters):
    ax.imshow(f.squeeze(), cmap="gray")
    ax.axis("off")
plt.suptitle("Learned first-layer 3x3 kernels", y=1.04)
plt.tight_layout()
plt.show()


### Feature maps — what the first layer *sees*

Each kernel, slid across one image, produces a **feature map**. We can inspect this by looking 
at what pattern the learned feature map "brightens".

In [ ]:
img, label = test_data[0]
with torch.no_grad():
    fmaps = F.relu(model.conv1(img.unsqueeze(0).to(device)))[0].cpu()   # (16, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, fm in zip(axes.flat, fmaps):
    ax.imshow(fm, cmap="gray")
    ax.axis("off")
plt.suptitle(f"conv1 feature maps for a '{class_names[label]}'", y=1.04)
plt.tight_layout()
plt.show()


### Does the convolution actually help? An MLP baseline

Finally, let's train a simple MLP with a **similar parameter budget**. It flattens the image
immediately, throwing away all spatial structure, so the CNN should win. 

(We can use the same `train` function as before since it does not care what model it gets.)

In [ ]:
class MLP(nn.Module):
    """A plain fully-connected baseline: flatten the image, then Linear layers.
    No convolutions, no parameter sharing, no notion of which pixels are neighbours."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                 # (B, 1, 28, 28) -> (B, 784)
            nn.Linear(28 * 28, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.net(x)

mlp = MLP().to(device)
print("CNN parameters:", f"{count_parameters(model):,}")
print("MLP parameters:", f"{count_parameters(mlp):,}")
print()

_ = train(mlp, train_loader, epochs=3, lr=1e-3)
print()
print(f"CNN test accuracy: {accuracy(model, test_loader):.3f}")
print(f"MLP test accuracy: {accuracy(mlp,   test_loader):.3f}")


## A small ethics moment

The same CNN architecture you just built also powers facial-analysis systems and computational photography. Take a look at Google's post on their "Real Tone" system, which was introduced in the Pixel phones to tackle the training biases that lead to poor photographic performance on those with darker skin tones: https://store.google.com/gb/magazine/inclusive-photography-real-tone

## Recap

**You built and trained a CNN.** You:

- Saw why MLP parameters explode on images, and how **parameter sharing** fixes it
- Predicted every layer's **shape** and **parameter count** by hand
- Trained with the **Lab 5 loop** over `DataLoader` batches, on the GPU, with cross-entropy
- Read the model's **confusion matrix**, filters and feature maps — and left an MLP behind

### Extensions (optional)

- Train for 10 epochs. Does test accuracy keep pace with train accuracy, or start to lag (overfitting)?
- Add a third conv block (32→64, padding 1, then pool). What does the flattened size become?
- Swap `MaxPool2d` for `AvgPool2d`. Does it matter here?

### Next time — Lab 7

Next we flip classification around and **generate** images: an **autoencoder** squeezes
an image to a handful of numbers and rebuilds it, then a **variational autoencoder
(VAE)** samples brand-new clothing. Today's convolutions are the building blocks
throughout.

## Solutions

Worked answers for every `# TODO` above. Have a genuine go first — including asking a
model, which will often get you there before this section does.

To pick up where you left off: copy the answer into the matching `# TODO` cell above and
re-run from there, so the rest of the notebook uses your version.

**Solution — the output-size formula**

No padding, so the 5×5 kernel cannot hang over the edge: ⌊(28 + 0 − 5)/1⌋ + 1 = **24**;
the pool then halves it to **12**. This is why the model pads its 3×3 convs — sizes stay
put until the pooling shrinks them.

In [ ]:
after_conv = (28 + 2*0 - 5) // 1 + 1    # 24
after_pool = (24 + 2*0 - 2) // 2 + 1    # 12
print(after_conv, after_pool)

**Solution — the CNN**

A chatbot may offer `nn.Sequential` instead, which also works — but the inspection cells
refer to `model.conv1`, `model.pool` and friends, so keep the spec's attribute names.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1,  out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))     # -> (B, 16, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))     # -> (B, 32,  7,  7)
        x = torch.flatten(x, 1)                  # -> (B, 1568)
        x = F.relu(self.fc1(x))                  # -> (B, 128)
        x = self.fc2(x)                          # -> (B, 10) logits
        return x

**Solution — the parameter counts**

Don't forget the biases — one per filter or output feature. The two convs hold under
5,000 parameters between them; `fc1` holds about 97% of the model. That is the
parameter-sharing payoff.

In [ ]:
conv1_expected = (3 * 3 * 1 + 1) * 16     # 160
conv2_expected = (3 * 3 * 16 + 1) * 32    # 4,640
fc1_expected   = 1568 * 128 + 128         # 200,832
fc2_expected   = 128 * 10 + 10            # 1,290
print("total:", conv1_expected + conv2_expected + fc1_expected + fc2_expected)

**Solution — the accuracy helper**

`argmax(dim=1)` picks the highest-scoring class per row. The `.item()` matters: it pulls
the plain Python number out of a one-element tensor, so `correct` stays an ordinary
int.

In [ ]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval()                          # evaluation mode
    correct, total = 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        preds = logits.argmax(dim=1)      # the class with the highest score
        correct += (preds == y).sum().item()
        total   += y.size(0)
    model.train()                         # back to training mode
    return correct / total


**Solution — the training loop**

The five middle lines are character-for-character the Lab 5 loop; everything around them
is bookkeeping.

In [ ]:
def train(model, train_loader, epochs=3, lr=1e-3):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_history = []
    for epoch in range(epochs):
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()                # 1. clear gradients
            logits = model(X)                    # 2. forward
            loss = loss_fn(logits, y)            # 3. loss
            loss.backward()                      # 4. backward
            optimizer.step()                     # 5. update
            loss_history.append(loss.item())
        print(f"epoch {epoch+1}/{epochs} | last loss {loss.item():.4f} "
              f"| train acc {accuracy(model, train_loader):.3f} "
              f"| test acc {accuracy(model, test_loader):.3f}")
    return loss_history

**Solution — predictions on a batch**

Three operations in one line: `.to(device)`, `argmax` over the class dimension, then
`.cpu()` so matplotlib can read the result.

In [ ]:
model.eval()
X, y = next(iter(test_loader))
with torch.no_grad():
    preds = model(X.to(device)).argmax(dim=1).cpu()
print(preds[:12])
print([class_names[p] for p in preds[:6]])

**Solution — the worst confusion**

Expect `Shirt` in the worst pair — confused with T-shirt/top, coat or pullover, the pairs
a human hesitates over too.

In [ ]:
off_diag = confusion.clone()
off_diag.fill_diagonal_(0)          # ignore the correct predictions
idx = off_diag.argmax().item()      # flat position in the 10x10 grid
t, p = divmod(idx, 10)
print(f"worst confusion: true '{class_names[t]}' predicted as '{class_names[p]}' "
      f"({off_diag[t, p].item()} times)")